In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/raw/data.csv")

df.head()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
numerical_cols = [
    "Amount",
    "Value",
    "CountryCode",
    "PricingStrategy"
]

for col in numerical_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(col)
    plt.show()

# Exploratory Data Analysis – Distribution Plots

## 1. Amount
- **X-axis**: Amount (ranges from 0.0 to ~1.0e7)
- **Y-axis**: Count (up to ~60,000)
- **Observation**: Highly skewed distribution; most values are concentrated near 0, with a long tail extending to very large amounts (1e7).

## 2. CountryCode
- **X-axis**: CountryCode (approximately 255.6 – 256.4)
- **Y-axis**: Count
- **Observation**: Narrow range with a peak near 256.0; likely a binned or encoded numeric variable, not standard integer country codes.

## 3. PricingStrategy
- **X-axis**: PricingStrategy (0.0 – 4.0)
- **Y-axis**: Count (0 – ~80,000)
- **Observation**: 
  - Multiple distinct clusters (e.g., near 0.0, 1.0, 2.0, 3.0, 4.0)
  - Highest frequency observed around 0.0 or 1.0
  - Some intermediate values (e.g., 0.5, 1.5, 2.5, 3.5) present but less frequent

## Summary
- `Amount` is heavily right-skewed with extreme outliers.
- `CountryCode` is tightly concentrated in a small numeric band.
- `PricingStrategy` appears categorical or discrete with several common values.

In [ ]:
categorical_cols = [
    "CurrencyCode",
    "ProviderId",
    "ProductCategory",
    "ChannelId"
]

for col in categorical_cols:
    plt.figure(figsize=(10,4))
    df[col].value_counts().head(10).plot(kind='bar')
    plt.title(col)
    plt.show()

# Categorical Variable Distributions

## 1. CurrencyCode
- **Y-axis**: Count (0 – 100,000)
- **Observation**: One currency (likely USD, labeled `US$`) dominates the dataset, with very high frequency compared to others.

## 2. ProviderId
- **Categories**: ProviderId_1 through ProviderId_6
- **Observation**: 
  - Providers appear in a specific order of frequency (descending: _4, _6, _5, _1, _3, _2)
  - Variation in counts across providers, suggesting uneven market share or usage.

## 3. ProductCategory
- **Categories**: 
  - financial_services
  - airtime
  - utility_bill
  - data_bundles
  - tv
  - ticket
  - movies
  - transport
  - other
- **Observation**: 
  - `financial_services` and `airtime` are likely the most frequent categories.
  - Wide variety of product types, ranging from digital goods (data, movies) to physical services (transport, tickets).

## 4.  ChannelId
- **Categories**: Channel3, Channel2, Channel5, Channel1
- **Observation**: 
  - Four distinct channels, with varying frequencies.
  - Channel naming not sequential by frequency (Channel3 and Channel2 appear most common).

## Summary
- **Currency** is heavily concentrated in one type (USD-equivalent).
- **ProviderId** and **ChannelId** show uneven distributions, indicating potential business concentration.
- **ProductCategory** covers both digital and utility services, with finance and airtime being prominent.

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr,
            annot=True,
            cmap='coolwarm')
plt.show()

## Key Insights

- **Strong positive correlations**:
  - `CountryCode` with `Amount` (0.99) – almost perfectly linear.
  - `Amount` with `FraudResult` (0.57) – moderate positive relationship.
  - `CountryCode` with `FraudResult` (0.56) – moderate positive relationship.

- **Weak or negligible correlations**:
  - `PricingStrategy` shows very weak negative correlations with all other variables (between -0.017 and -0.062).

- **Conclusion**:
  - `Amount` and `CountryCode` are highly related and both moderately associated with `FraudResult`.
  - `PricingStrategy` appears largely independent of the other numeric features.

In [ ]:
for col in ["Amount","Value"]:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()

In [ ]:
df["TransactionStartTime"] = pd.to_datetime(
    df["TransactionStartTime"]
)

In [ ]:
df["Hour"] = df["TransactionStartTime"].dt.hour

sns.histplot(df["Hour"])
plt.show()

### Distribution of Transactions by Hour

- **Peak activity**: Hours 10–12 (late morning to noon), with over 6,000 transactions.
- **Secondary peak**: Hours 14–16 (early to mid-afternoon).
- **Lowest activity**: Hours 0–5 (late night to early morning), with near-zero transaction counts.
- **Pattern**: Strong daytime concentration, minimal overnight activity.

### Implication for Credit Risk Modeling

- Customers who transact primarily during **regular business hours** may exhibit more stable, predictable financial behavior.
- Customers with **late-night or irregular-hour transactions** could represent higher-risk segments, depending on other behavioral indicators (e.g., frequency, monetary value).
- The `Hour` feature will be included in the model pipeline alongside other behavioral features (e.g., RFM metrics) to improve predictive power for the proxy `is_high_risk` target variable.

### Alignment with Basel II Requirements

- Extracting interpretable features like `Hour` supports **model transparency** and **documentation**, which are essential under Basel II’s emphasis on risk measurement and interpretability.
- This feature, when combined with Weight of Evidence (WoE) transformation, helps maintain a **defensible link between behavior and credit risk**.